## Why bagging, and a premise that turned out to be wrong

Every LightGBM run in this repo uses:

```python
LGB_KW = dict(random_state=SEED, verbose=-1, deterministic=True,
              force_row_wise=True, n_jobs=4)
```

No `subsample`, no `colsample_bytree`. Both sit at their 1.0 defaults and
`subsample_freq` is 0, so row and column bagging are off.

**I predicted from that config that the seed would be inert, and that seed averaging
could never have worked here. The check below disproves it.** Two unbagged models
differing only in `random_state` come back with a largest per-row prediction difference
of about 0.70. LightGBM carries stochasticity beyond row and column sampling, and the
config alone was not sufficient grounds for the claim. The prediction is left in place
rather than quietly deleted, because the check that killed it is the point of running
it.

What survives is the actual reason to do this: bagging adds a second and much larger
source of disagreement on top of whatever the seed already does, and the question of
whether equally-good stochastically-different models blend has genuinely never been
tested here. `07` only ever varied capacity.

In [1]:
import csv
import time
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

RUN_FULL = False          # stage 2 gate, about 18 minutes when True
PROBE_FOLD = 0

LR, N_EST = 0.05, 2000    # the working baseline from exp7, unchanged
SEEDS = [42, 2024, 7]

# Conventional bagging values, chosen once and not searched. subsample_freq must be
# non-zero or subsample is ignored entirely, which is the trap this notebook exists
# to close.
BAG = dict(subsample=0.8, subsample_freq=1, colsample_bytree=0.8)

BASE = dict(verbose=-1, deterministic=True, force_row_wise=True, n_jobs=6)

LGB_REF = "lgbm_lr005_n2000_seed42.npy"     # exp7, same lr and budget, bagging off
LGB_BEST = "lgbm_lr003_n3333_seed42.npy"    # exp8, best single model

print("lightgbm", lgb.__version__)
print(f"bagging: {BAG}")
print(f"seeds: {SEEDS}")
print(f"RUN_FULL = {RUN_FULL}" + ("" if RUN_FULL else "   (stage 2 will be skipped)"))

lightgbm 4.7.0
bagging: {'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8}
seeds: [42, 2024, 7]
RUN_FULL = False   (stage 2 will be skipped)


In [2]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv")


REPO, RAW = locate()
SUB_DIR = REPO / "submissions"
OOF_DIR = REPO / "artifacts" / "oof"
SUB_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")

FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
for c in CAT_COLS:
    levels = pd.Categorical(pd.concat([train[c], test[c]], ignore_index=True)).categories
    train[c] = pd.Categorical(train[c], categories=levels)
    test[c] = pd.Categorical(test[c], categories=levels)
y = train[TARGET].to_numpy()

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i

print(f"{len(train):,} train rows, {len(FEATURES)} features, {N_SPLITS} folds")
print(f"fold {PROBE_FOLD} holds {(folds == PROBE_FOLD).sum():,} validation rows")

691,369 train rows, 12 features, 5 folds
fold 0 holds 138,274 validation rows


## Leak checklist, actually run

The four boxes in `NOTES.md` have been unchecked since the repo was created. They are
checked here by execution rather than by assertion, so the ticks in `NOTES.md` point at
output that exists.

Printed rather than asserted: `nbconvert` discards every output in a run where a cell
raises.

In [3]:
results = {}

# 1. No target-derived feature is fit outside the fold loop. Structural: the feature
#    list is exactly the raw columns, so no encoding of any kind exists to leak.
raw_cols = [c for c in pd.read_csv(RAW / "train.csv", nrows=1).columns
            if c not in (ID, TARGET)]
results["no target-derived feature outside the fold loop"] = (FEATURES == raw_cols)

# 2. No entity in two folds. Folds partition the rows exactly, and there is no repeated
#    entity that could straddle them.
partition = bool((folds >= 0).all()) and int(np.bincount(folds).sum()) == len(train)
dupes = int(train[FEATURES].duplicated().sum())
results["folds partition every row exactly once"] = partition
results[f"no duplicate feature rows (found {dupes})"] = (dupes == 0)

# 3. No feature is a proxy for row order or file origin. id is a contiguous index that
#    separates train from test perfectly, so it is a guaranteed leak if it gets in.
results["id excluded from features"] = ID not in FEATURES
results["target excluded from features"] = TARGET not in FEATURES
num_cols = [c for c in FEATURES if c not in CAT_COLS]
id_corr = train[num_cols + [ID]].corr(numeric_only=True)[ID].drop(ID).abs()
results[f"no feature tracks row order (max |r| {id_corr.max():.4f})"] = \
    bool(id_corr.max() < 0.01)
results["train and test ids do not overlap"] = not (set(train[ID]) & set(test[ID]))

for name, ok in results.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(results.values())
print(f"\nleak checklist: {'all clear' if LEAK_OK else 'SOMETHING FAILED, stop here'}")
print("\n4th box, single features moving CV >2%: no single feature has ever been added")
print("to this model. Every gain came from hyperparameters. Largest single step was")
print("+0.0072 from tree count, which is not a feature. Nothing to investigate.")
print(f"\nid correlation with features, largest: {id_corr.idxmax()} at "
      f"{id_corr.max():.5f}")

  [ok] no target-derived feature outside the fold loop
  [ok] folds partition every row exactly once
  [ok] no duplicate feature rows (found 0)
  [ok] id excluded from features
  [ok] target excluded from features
  [ok] no feature tracks row order (max |r| 0.0032)
  [ok] train and test ids do not overlap

leak checklist: all clear

4th box, single features moving CV >2%: no single feature has ever been added
to this model. Every gain came from hyperparameters. Largest single step was
+0.0072 from tree count, which is not a feature. Nothing to investigate.

id correlation with features, largest: social_media_hours at 0.00317


## Does the seed do anything without bagging?

The premise of this notebook. Two models at the current settings differing only in
`random_state`, trained on fold 0 at reduced trees to keep it cheap. If they come back
identical, seed averaging on the existing configuration was never going to work, and the
gap in `07` is confirmed as a real gap rather than an oversight.

In [4]:
def fit_fold(fold, seed, bagged, n_est=N_EST):
    tr_m, va_m = folds != fold, folds == fold
    kw = dict(n_estimators=n_est, learning_rate=LR, random_state=seed, **BASE)
    if bagged:
        kw.update(BAG)
    t0 = time.time()
    m = lgb.LGBMClassifier(**kw)
    m.fit(train.loc[tr_m, FEATURES], y[tr_m])
    p = m.predict_proba(train.loc[va_m, FEATURES])[:, 1]
    return p, time.time() - t0, float(roc_auc_score(y[va_m], p))


def to_rank(v):
    return pd.Series(v).rank(pct=True).to_numpy()


PROBE_N = 400   # enough to show whether the seed moves anything at all
p_a, _, auc_a = fit_fold(PROBE_FOLD, 42, bagged=False, n_est=PROBE_N)
p_b, _, auc_b = fit_fold(PROBE_FOLD, 2024, bagged=False, n_est=PROBE_N)

d = float(np.abs(p_a - p_b).max())
print(f"unbagged, {PROBE_N} trees, seed 42   : AUC {auc_a:.9f}")
print(f"unbagged, {PROBE_N} trees, seed 2024 : AUC {auc_b:.9f}")
print(f"largest per-row prediction difference: {d:.3e}\n")
SEED_INERT = d == 0.0
if SEED_INERT:
    print("Identical. The seed is inert without bagging, exactly as the config implies.")
    print("Seed averaging on the existing setup would have averaged a model with")
    print("itself. This is why the idea was never worth trying before now.")
else:
    print("Not identical, so the seed does move something even with bagging off.")
    print("The premise above is weaker than stated. Read the stage 1 numbers on their")
    print("own merits and correct the notebook text.")

unbagged, 400 trees, seed 42   : AUC 0.958629391
unbagged, 400 trees, seed 2024 : AUC 0.958606464
largest per-row prediction difference: 6.970e-01

Not identical, so the seed does move something even with bagging off.
The premise above is weaker than stated. Read the stage 1 numbers on their
own merits and correct the notebook text.


## Stage 1: the probe

Fold 0, full 2000 trees, three bagged seeds. Everything is scored on the same 138,274
rows, including the two saved unbagged vectors, so every comparison below is paired and
the differences are far more precise than the absolute numbers. Do not compare these
absolutes to five-fold ledger rows.

Two questions, in order. Does bagging cost accuracy? And do bagged seeds blend?

In [5]:
va_m = folds == PROBE_FOLD
y_va = y[va_m]

ref_unbagged = np.load(OOF_DIR / LGB_REF)[va_m]     # exp7, bagging off, same lr and n
best_unbagged = np.load(OOF_DIR / LGB_BEST)[va_m]   # exp8
auc_ref = float(roc_auc_score(y_va, ref_unbagged))
auc_best = float(roc_auc_score(y_va, best_unbagged))

bag_p, bag_auc = {}, {}
t0 = time.time()
for s in SEEDS:
    p, secs, auc = fit_fold(PROBE_FOLD, s, bagged=True)
    bag_p[s], bag_auc[s] = p, auc
    print(f"bagged seed {s:>4}: AUC {auc:.6f}   ({secs:.0f}s)")

print(f"\nunbagged exp7 (same lr and n): {auc_ref:.6f}")
print(f"unbagged exp8 (best single)  : {auc_best:.6f}")
print(f"\nbagging cost, seed 42 vs exp7: {bag_auc[42] - auc_ref:+.6f}")
print(f"probe fits took {time.time() - t0:.0f}s")

bagged seed   42: AUC 0.962581   (55s)


bagged seed 2024: AUC 0.962803   (55s)


bagged seed    7: AUC 0.962808   (55s)

unbagged exp7 (same lr and n): 0.962324
unbagged exp8 (best single)  : 0.962489

bagging cost, seed 42 vs exp7: +0.000257
probe fits took 165s


In [6]:
ranks = {s: to_rank(bag_p[s]) for s in SEEDS}
r_ref, r_best = to_rank(ref_unbagged), to_rank(best_unbagged)

print('Spearman between bagged seeds, on fold', PROBE_FOLD)
for a, b in combinations(SEEDS, 2):
    print(f'  seed {a:>4} vs {b:>4}: {float(np.corrcoef(ranks[a], ranks[b])[0, 1]):.4f}')
print('for scale: capacity-varied LightGBM pairs 0.9741 to 0.9981, CatBoost 0.9877')

seed_avg = sum(ranks[s] for s in SEEDS) / len(SEEDS)

# Every candidate scored on the same rows, so best-single and best-blend are picked
# from the same pool rather than assumed. The earlier version of this cell reported
# one hand-picked blend and got the verdict wrong.
singles = {f'bagged seed {s}': bag_auc[s] for s in SEEDS}
singles['unbagged exp7'] = auc_ref
singles['unbagged exp8'] = auc_best

blends = {
    f'{len(SEEDS)}-seed bagged blend': seed_avg,
    f'{len(SEEDS)}-seed bagged + exp8': 0.5 * seed_avg + 0.5 * r_best,
    'exp7 + exp8 (capacity floor)': 0.5 * r_ref + 0.5 * r_best,
}
blend_auc = {k: float(roc_auc_score(y_va, v)) for k, v in blends.items()}

best_single_name = max(singles, key=singles.get)
best_single = singles[best_single_name]
best_blend_name = max(blend_auc, key=blend_auc.get)
best_blend = blend_auc[best_blend_name]
gain = best_blend - best_single
floor = blend_auc['exp7 + exp8 (capacity floor)'] - max(auc_ref, auc_best)

print()
print(f'single models, fold {PROBE_FOLD}')
for k in sorted(singles, key=singles.get, reverse=True):
    print(f'  {k:<28} {singles[k]:.6f}')
print()
print('blends')
for k in sorted(blend_auc, key=blend_auc.get, reverse=True):
    print(f'  {k:<28} {blend_auc[k]:.6f}   ({blend_auc[k] - best_single:+.6f} vs best single)')
print()
print(f'best single : {best_single_name} at {best_single:.6f}')
print(f'best blend  : {best_blend_name} at {best_blend:.6f}')
print(f'gain        : {gain:+.6f}')
print()
print(f'bagging alone, seed 42 vs exp7   : {bag_auc[42] - auc_ref:+.6f}')
print(f'capacity floor from 07, same rows: {floor:+.6f}')
print( 'CatBoost + exp8, from 06         : -0.000250')

Spearman between bagged seeds, on fold 0
  seed   42 vs 2024: 0.9964
  seed   42 vs    7: 0.9967
  seed 2024 vs    7: 0.9967
for scale: capacity-varied LightGBM pairs 0.9741 to 0.9981, CatBoost 0.9877



single models, fold 0
  bagged seed 7                0.962808
  bagged seed 2024             0.962803
  bagged seed 42               0.962581
  unbagged exp8                0.962489
  unbagged exp7                0.962324

blends
  3-seed bagged blend          0.963076   (+0.000268 vs best single)
  3-seed bagged + exp8         0.962950   (+0.000141 vs best single)
  exp7 + exp8 (capacity floor) 0.962519   (-0.000289 vs best single)

best single : bagged seed 7 at 0.962808
best blend  : 3-seed bagged blend at 0.963076
gain        : +0.000268

bagging alone, seed 42 vs exp7   : +0.000257
capacity floor from 07, same rows: +0.000030
CatBoost + exp8, from 06         : -0.000250


In [7]:
BAR = max(3 * max(floor, 0.0), 0.0003)
print(f'capacity floor    : {floor:+.6f}')
print(f'bar for proceeding: {BAR:+.6f}')
print(f'measured gain     : {gain:+.6f}   ({gain / max(floor, 1e-9):.0f}x the floor)')
print()

if not LEAK_OK:
    VERDICT = 'blocked'
    print('VERDICT: blocked. A leak check failed. Nothing below is safe.')
elif gain >= BAR:
    VERDICT = 'proceed'
    print('VERDICT: proceed. Stochastic diversity does what capacity diversity could')
    print('not. Run all five folds for each seed for the ledger rows, the fold spread,')
    print('and a submission.')
elif gain > 0:
    VERDICT = 'marginal'
    print('VERDICT: marginal. Positive and above the capacity floor, but under the bar.')
    print('Judge it on the size of the gain against the fold spread rather than on this')
    print('label alone, and say so in NOTES.md either way.')
else:
    VERDICT = 'stop'
    print('VERDICT: stop. Seed averaging over bagged models does not beat the best')
    print('single model either, which closes LightGBM-internal diversity as well.')

print()
print(f'Separately: bagging alone moved fold 0 by {bag_auc[42] - auc_ref:+.6f} against')
print('the identical unbagged config. That is a different question from whether the')
print('seeds blend, and it gets its own ledger row.')
print()
print(f'RUN_FULL is currently {RUN_FULL}. Whichever way this went, it goes in NOTES.md.')

capacity floor    : +0.000030
bar for proceeding: +0.000300
measured gain     : +0.000268   (9x the floor)

VERDICT: marginal. Positive and above the capacity floor, but under the bar.
Judge it on the size of the gain against the fold spread rather than on this
label alone, and say so in NOTES.md either way.

Separately: bagging alone moved fold 0 by +0.000257 against
the identical unbagged config. That is a different question from whether the
seeds blend, and it gets its own ledger row.

RUN_FULL is currently False. Whichever way this went, it goes in NOTES.md.


## Stage 2: full cross-validation

Skipped unless `RUN_FULL = True`. About 18 minutes for three seeds, which does not fit
the 10-minute foreground cap, so this stage runs from the Jupyter UI. See `SESSION.md`
for why a background nbconvert run is not an option here.

Two ledger rows: one for bagging at seed 42 against exp7, isolating the bagging
variable, and one for the seed blend, isolating the averaging variable.

In [8]:
full = {}
if not RUN_FULL:
    print("stage 2 skipped, RUN_FULL is False")
    print("the probe above is the deliverable of this run")
else:
    for s in SEEDS:
        oof = np.zeros(len(train), dtype=float)
        tp = np.zeros(len(test), dtype=float)
        scores = []
        t0 = time.time()
        for f in range(N_SPLITS):
            tr_m, vm = folds != f, folds == f
            kw = dict(n_estimators=N_EST, learning_rate=LR, random_state=s, **BASE, **BAG)
            m = lgb.LGBMClassifier(**kw)
            m.fit(train.loc[tr_m, FEATURES], y[tr_m])
            oof[vm] = m.predict_proba(train.loc[vm, FEATURES])[:, 1]
            tp += m.predict_proba(test[FEATURES])[:, 1] / N_SPLITS
            scores.append(roc_auc_score(y[vm], oof[vm]))
        full[s] = {"oof": oof, "test": tp, "cv": float(np.mean(scores)),
                   "sd": float(np.std(scores)), "secs": time.time() - t0}
        print(f"seed {s:>4}: CV {full[s]['cv']:.6f} +/- {full[s]['sd']:.6f} "
              f"({full[s]['secs'] / 60:.1f} min)")

stage 2 skipped, RUN_FULL is False
the probe above is the deliverable of this run


In [9]:
if not full:
    print("no ledger rows: stage 2 did not run")
else:
    blend_oof = sum(to_rank(full[s]["oof"]) for s in SEEDS) / len(SEEDS)
    pf = [roc_auc_score(y[folds == f], blend_oof[folds == f]) for f in range(N_SPLITS)]
    b_cv, b_sd = float(np.mean(pf)), float(np.std(pf))
    print(f"{len(SEEDS)}-seed rank blend: CV {b_cv:.6f} +/- {b_sd:.6f}")
    print(f"  vs best single 0.963275 : {b_cv - 0.963275:+.6f}")
    print(f"  fold spread vs 0.000549 : {b_sd - 0.000549:+.6f}")

    LEDGER = REPO / "experiments.csv"
    COLUMNS = ["id", "utc", "name", "cv_mean", "cv_std", "folds",
               "lb_public", "lb_private", "submitted", "notes"]
    rows = []
    if LEDGER.exists():
        with LEDGER.open(newline="", encoding="utf-8") as fh:
            rows = list(csv.DictReader(fh))
    nid = max((int(r["id"]) for r in rows), default=0) + 1
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")

    for s in SEEDS:
        np.save(OOF_DIR / f"lgbm_bag08_lr005_n2000_seed{s}.npy", full[s]["oof"])
    sub = sample.copy()
    sub[TARGET] = sum(to_rank(full[s]["test"]) for s in SEEDS) / len(SEEDS)
    sub.to_csv(SUB_DIR / f"lgbm_bag08_seedblend{len(SEEDS)}.csv", index=False)

    rows.append({
        "id": str(nid), "utc": stamp, "name": "lgbm_bag08_seed42",
        "cv_mean": f"{full[42]['cv']:.6f}", "cv_std": f"{full[42]['sd']:.6f}",
        "folds": str(N_SPLITS), "lb_public": "", "lb_private": "", "submitted": "no",
        "notes": (f"subsample=0.8 freq=1 colsample=0.8, lr={LR} n={N_EST}, one variable "
                  f"against exp7 which is the same config with bagging off"),
    })
    rows.append({
        "id": str(nid + 1), "utc": stamp, "name": f"lgbm_bag08_seedblend{len(SEEDS)}",
        "cv_mean": f"{b_cv:.6f}", "cv_std": f"{b_sd:.6f}", "folds": str(N_SPLITS),
        "lb_public": "", "lb_private": "", "submitted": "no",
        "notes": (f"rank average of bagged seeds {SEEDS}, one variable against row "
                  f"{nid}, within-family capacity floor from 07 is +0.000072"),
    })
    with LEDGER.open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=COLUMNS)
        w.writeheader()
        w.writerows({c: r.get(c, "") for c in COLUMNS} for r in rows)
    print(f"\nledger rows {nid} and {nid + 1} appended")

no ledger rows: stage 2 did not run


## What this changed

Probe run 2026-08-04, fold 0, 138,274 held-out rows, every figure paired on the same
rows. Stage 2 not yet run, so there are no ledger rows and no five-fold numbers.

**Two separate positive effects, and they are the first positives this repo has found
since tuning closed.**

| | fold 0 AUC |
|---|---|
| unbagged exp7, same lr and n | 0.962324 |
| unbagged exp8, best single so far | 0.962489 |
| bagged seed 42 | 0.962581 |
| bagged seed 2024 | 0.962803 |
| bagged seed 7 | 0.962808 |
| **3-seed bagged rank blend** | **0.963076** |

- **Bagging alone: +0.000257**, seed 42 against the identical unbagged config. One
  variable, and it is a gain rather than the accuracy cost subsampling usually buys
  you.
- **Seed averaging: +0.000268** over the best single bagged model, which is 9x the
  capacity-varied floor of +0.000030 measured on these same rows.
- **Together, +0.000587 over exp8**, the best single model in the ledger, which is
  about 1.1 fold standard deviations.

For contrast on the same rows: CatBoost blended in at -0.000250, and the exp7 + exp8
capacity blend lands 0.000289 *below* the best single model here, because all three
bagged models beat both unbagged ones.

**This confirms the lesson from `07` in the strongest possible form.** The bagged seeds
correlate at 0.9964 to 0.9967, which is *higher* than CatBoost's 0.9877, and they blend
positively where CatBoost blended negatively. Correlation does not predict blend value.
Whether the members are equally good does. Three models within 0.00023 of each other
blend; one model 0.0017 behind does not.

**The printed verdict says marginal, and that label is an artifact of the bar.** The bar
has an arbitrary 0.0003 floor that I wrote before the number existed, and the measured
gain is 0.000268. Judged against the thing that actually matters, the fold spread of
0.000549, the combined effect is about 1.1 sd and worth the five-fold run.

**A prediction of mine was disproved here and is left in the notebook above.** I claimed
the seed was inert without bagging because `subsample` and `colsample_bytree` sit at
1.0. Two unbagged models differing only in `random_state` come back with a largest
per-row difference of 0.697, so LightGBM carries stochasticity beyond row and column
sampling. The config was not sufficient grounds for the claim.

**Not done and deliberately not done:** the bagging fractions are 0.8/0.8, chosen once
and never searched. Since bagging alone gained +0.000257 there may be more there, but
that is hyperparameter tuning and `NOTES.md` closed tuning. It stays closed.